In [1]:
import pandas as pd
from pprint import pprint
import re
import numpy as np

pd.set_option('display.max_rows', None)

In [2]:
!mkdir -p tables

In [3]:
def load_results(filename, roundat=2):
    results = pd.read_pickle(filename)
    results['batch_size']=results.train_params.str.extract(".*batch_size=([0-9]+).*")
    results['learning_rate']=results.train_params.str.extract(".*learning_rate=(.+?),.*")
    print("Number of results so far: ", len(results))

    results['f1_test'] = results.f1_test.round(roundat)
    results['PER_f1_test'] = results.PER_f1_test.round(roundat)
    results['LOC_f1_test'] = results.LOC_f1_test.round(roundat)
    results['ORG_f1_test'] = results.ORG_f1_test.round(roundat)

    return results

In [4]:
#Make a nice table.

def merge_historical_results(rsingle, rzefys, rall):
    _res_single = rsingle
    _res_pretrained_zefys = rzefys.drop(columns=['pretrain'])
    _res_pretrained_all = rall.drop(columns=['pretrain'])
    
    merged_results = _res_single.merge(_res_pretrained_zefys, on=['train', 'test','model'], suffixes=['','_PZ'], how='outer')
    merged_results = merged_results.merge(_res_pretrained_all, on=['train', 'test','model'], suffixes=['','_PA'], how='outer')

    tmp = pd.concat([rsingle, rzefys, rall])

    per_loc_org=[]
    for _, res in tmp.groupby(['train', 'test', 'model']):
        per_loc_org.append(res.sort_values('f1_test', ascending=False).iloc[[0]])

    per_loc_org=pd.concat(per_loc_org).rename(columns={'PER_f1_test': 'PER_f1_best', 'LOC_f1_test': 'LOC_f1_best', 'ORG_f1_test': 'ORG_f1_best'})

    print(per_loc_org.columns)
    
    per_loc_org=per_loc_org[['train', 'test','model', 'PER_f1_best', 'LOC_f1_best', 'ORG_f1_best']]

    merged_results = merged_results.merge(per_loc_org, on=['train', 'test','model'])

    return merged_results

def make_nice_table(df_input, dataset_order, use_columns, max_columns, dataset_column='test', additional_columns=['dataset', 'model']):
    
    atab = df_input.copy()
    
    atab['max_f1'] = atab[max_columns].apply(lambda x: x.mean(), axis=1)
    atab = atab.sort_values(['test', 'max_f1'], ascending=[False,False])
    
    atab['dataset'] = atab[dataset_column] 
    
    atab = pd.concat([atab.loc[atab.dataset==o] for o in dataset_order]).reset_index(drop=True)

    def model_rename(x):
        if x=="electra-base-german-europeana-cased-discriminator":
            return "Europ-Electra"            
            
        if x=="bert-base-historic-multilingual-cased":
            return "hmBERT"            
                        
        if x=="bert-tiny-historic-multilingual-cased":
            return "hmBERT-tiny"            
                        
        if x=="bert-mini-historic-multilingual-cased":
            return "hmBERT-mini"            
                        
        if x=="bert-base-german-cased":
            return "GermanBERT"            
                        
        if x=="roberta-base":
            return "Roberta"            
                        
        if x=="xlm-roberta-base":
            return "XLM-Roberta"            
                        
        if x=="gbert-base":
            return "GBERT"            
                        
        if x=="distilbert-base-uncased":
            return "Distilbert"            
                        
    
    atab['model'] = atab['model'].apply(model_rename)
    
    #def abbreviate(x):
    #    return "".join([p[0] for p in x.split('-')]).upper()
    # 
    #atab['model'] = atab['model'].apply(abbreviate)
    
    def dataset_rename(x):
    
        if x=="zefys2025":
            return "Anon2025"
    
        if x=="hipe2020":
            return "HIPE 2020"
    
        if x=="hisgerman":
            return "HisGermanNER"
    
        if x=="europeana-onb":
            return "Europeana-ÖNB"
    
        if x=="europeana-lft":
            return "Europeana-LFT"
    
        if x=="neiss-arendt":
            return "Neiss-Arendt"
    
        if x=="neiss-sturm":
            return "Neiss-Sturm"

        if x=='conll2003':
            return "CoNLL-2003"
            
        if x=='germeval_14':
            return "GermEval2014"
        
        return x
    
    atab['dataset'] = atab['dataset'].apply(dataset_rename)
    atab[dataset_column] = atab[dataset_column].apply(dataset_rename)
    atab['test'] = atab['test'].apply(dataset_rename)
    
    atab = atab[additional_columns + use_columns]

    btab=[]
    for dataset, part in atab.groupby('dataset', sort=False):

        part = part.drop(columns=['dataset'])
        btab.append(pd.concat([pd.DataFrame([("__" + dataset + "__",) + (part.shape[1]-1)*(np.nan,)], columns=part.columns), part],));

    btab = pd.concat(btab).reset_index(drop=True)
    
    nice_table = btab.style.\
        highlight_max(axis = 1, subset=max_columns, props='font-weight:bold').\
        format('{:.2f}', na_rep='-', subset=use_columns).\
        hide(axis='index')

    return nice_table, btab


def write_to_latex_file(nice_table, filename, num_cols):
    tex_code = nice_table.to_latex(hrules=True)

    tex_code = tex_code.replace("\\font-weightbold", "\\bf")

    tex_code = re.sub(r".*__(.*?)__.*", r"\\multicolumn{" + str(num_cols) + r"}{c}{\1} \\\\", tex_code)

    
    tex_code = re.sub(r"([^\s]*_[^\s]*)", r"$\1$", tex_code)
    tex_code = re.sub(r"_([^\s^_^$]*)", r"_{\1}", tex_code)
    
    with open(filename, 'w') as f:
        f.write(tex_code)

    return tex_code


# Contemporary

In [5]:
atab = []
for dataset, res in load_results('experiments/results-single.pkl').groupby(['train', 'test', 'model']):
    atab.append(res.sort_values('f1_test', ascending=False).iloc[[0]])
atab = pd.concat(atab).sort_values(['test','f1_early'], ascending=[True, False]).reset_index(drop=True)
atab = atab.loc[atab.train==atab.test]

test_val_diff = (atab['f1_test'] - atab['f1_early']).abs()
print('Average distance between early stopping f_1-score and test f_1-score: {}+-{}'.\
      format(test_val_diff.mean().round(2), test_val_diff.std().round(2)))

contemp_table, df_contemp = make_nice_table(atab,dataset_order=['conll2003', 'GermEval'], 
                                            use_columns=['f1_test', 'f1_early' , 'PER_f1_test', 'LOC_f1_test', 'ORG_f1_test'], max_columns=[])
contemp_table

Number of results so far:  6597
Average distance between early stopping f_1-score and test f_1-score: 0.02+-0.02


model,f1_test,f1_early,PER_f1_test,LOC_f1_test,ORG_f1_test
__CoNLL-2003__,-,-,-,-,-
Roberta,0.94,0.97,0.96,0.93,0.92
XLM-Roberta,0.93,0.96,0.96,0.92,0.90
GermanBERT,0.90,0.94,0.91,0.91,0.88
Distilbert,0.92,0.94,0.96,0.91,0.88
GBERT,0.90,0.94,0.92,0.90,0.88
hmBERT,0.89,0.93,0.92,0.90,0.84
Europ-Electra,0.88,0.91,0.90,0.90,0.84
hmBERT-mini,0.84,0.89,0.86,0.86,0.80
hmBERT-tiny,0.77,0.84,0.80,0.79,0.72


In [6]:
write_to_latex_file(contemp_table,'tables/contemporary.tex', df_contemp.shape[1])

'\\begin{tabular}{lrrrrr}\n\\toprule\nmodel & $f1_{test}$ & $f1_{early}$ & $PER_{f1}_{test}$ & $LOC_{f1}_{test}$ & $ORG_{f1}_{test}$ \\\\\n\\midrule\n\\multicolumn{6}{c}{CoNLL-2003} \\\\\nRoberta & 0.94 & 0.97 & 0.96 & 0.93 & 0.92 \\\\\nXLM-Roberta & 0.93 & 0.96 & 0.96 & 0.92 & 0.90 \\\\\nGermanBERT & 0.90 & 0.94 & 0.91 & 0.91 & 0.88 \\\\\nDistilbert & 0.92 & 0.94 & 0.96 & 0.91 & 0.88 \\\\\nGBERT & 0.90 & 0.94 & 0.92 & 0.90 & 0.88 \\\\\nhmBERT & 0.89 & 0.93 & 0.92 & 0.90 & 0.84 \\\\\nEurop-Electra & 0.88 & 0.91 & 0.90 & 0.90 & 0.84 \\\\\nhmBERT-mini & 0.84 & 0.89 & 0.86 & 0.86 & 0.80 \\\\\nhmBERT-tiny & 0.77 & 0.84 & 0.80 & 0.79 & 0.72 \\\\\n\\multicolumn{6}{c}{GermEval} \\\\\nGermanBERT & 0.88 & 0.89 & 0.93 & 0.90 & 0.79 \\\\\nGBERT & 0.88 & 0.89 & 0.94 & 0.89 & 0.78 \\\\\nXLM-Roberta & 0.86 & 0.87 & 0.91 & 0.88 & 0.76 \\\\\nEurop-Electra & 0.84 & 0.84 & 0.90 & 0.87 & 0.73 \\\\\nhmBERT & 0.82 & 0.83 & 0.88 & 0.86 & 0.69 \\\\\nRoberta & 0.81 & 0.83 & 0.87 & 0.83 & 0.71 \\\\\nDistilbert

In [7]:
atab = []
for dataset, res in load_results('experiments/results-single.pkl').groupby(['train', 'test','model']):
    atab.append(res.sort_values('f1_test', ascending=False).iloc[[0]])
atab = pd.concat(atab).sort_values(['test','f1_early'], ascending=[True, False]).reset_index(drop=True)
atab = atab.loc[atab.test==atab.train].reset_index(drop=True)

test_val_diff = (atab['f1_test'] - atab['f1_early']).abs()
print('Average distance between early stopping f_1-score and test f_1-score: {}+-{}'.\
      format(test_val_diff.mean().round(2), test_val_diff.std().round(2)))

res_single = atab


res_single[['test','train', 'model', 'epoch', 'f1_test', 'f1_early', 'PER_f1_test','LOC_f1_test', 'ORG_f1_test', 'learning_rate', 'batch_size']]

Number of results so far:  6597
Average distance between early stopping f_1-score and test f_1-score: 0.02+-0.02


,test,train,model,epoch,f1_test,f1_early,PER_f1_test,LOC_f1_test,ORG_f1_test,learning_rate,batch_size
0,GermEval,GermEval,bert-base-german-cased,4,0.88,0.89,0.93,0.90,0.79,2e-05,32
1,GermEval,GermEval,gbert-base,2,0.88,0.89,0.94,0.89,0.78,4e-05,64
2,GermEval,GermEval,xlm-roberta-base,3,0.86,0.87,0.91,0.88,0.76,2e-05,32
3,GermEval,GermEval,electra-base-german-europeana-cased-discriminator,3,0.84,0.84,0.90,0.87,0.73,4e-05,32
4,GermEval,GermEval,bert-base-historic-multilingual-cased,4,0.82,0.83,0.88,0.86,0.69,4e-05,64
5,GermEval,GermEval,roberta-base,4,0.81,0.83,0.87,0.83,0.71,2e-05,32
6,GermEval,GermEval,distilbert-base-uncased,6,0.76,0.77,0.80,0.80,0.66,0.0001,96
7,GermEval,GermEval,bert-mini-historic-multilingual-cased,10,0.72,0.75,0.78,0.76,0.59,4e-05,32
8,GermEval,GermEval,bert-tiny-historic-multilingual-cased,8,0.63,0.66,0.66,0.69,0.50,0.0001,32
9,conll2003,conll2003,roberta-base,4,0.94,0.97,0.96,0.93,0.92,2e-05,96


In [8]:
atab = []
for dataset, res in load_results('experiments/results-single.pkl').groupby(['train', 'test','model']):
    atab.append(res.sort_values('f1_test', ascending=False).iloc[[0]])
atab = pd.concat(atab).sort_values(['f1_test','f1_early'], ascending=[False, False]).reset_index(drop=True)
atab = atab.loc[atab.train=="zefys2025"]

test_val_diff = (atab['f1_test'] - atab['f1_early']).abs()
print('Average distance between early stopping f_1-score and test f_1-score: {}+-{}'.\
      format(test_val_diff.mean().round(2), test_val_diff.std().round(2)))

atab_zefys = atab.loc[(atab.test=="zefys2025") & (atab.train=="zefys2025")]
test_val_diff = (atab_zefys['f1_test'] - atab_zefys['f1_early']).abs()
print('ZEFYS2025 - Average distance between early stopping f_1-score and test f_1-score: {}+-{}'.\
      format(test_val_diff.mean().round(2), test_val_diff.std().round(2)))

btab = []
for data, res in atab.groupby('test', sort=False):
    btab.append(res.iloc[[0]])
btab = pd.concat(btab)    

#res_single[['test','train', 'model', 'epoch', 'f1_test', 'f1_early', 'PER_f1_test','LOC_f1_test', 'ORG_f1_test', 'learning_rate', 'batch_size']]
btab[['test','train', 'model', 'epoch', 'f1_test', 'f1_early', 'PER_f1_test','LOC_f1_test', 'ORG_f1_test', 'learning_rate', 'batch_size']]

Number of results so far:  6597
Average distance between early stopping f_1-score and test f_1-score: 0.17+-0.09
ZEFYS2025 - Average distance between early stopping f_1-score and test f_1-score: 0.01+-0.01


,test,train,model,epoch,f1_test,f1_early,PER_f1_test,LOC_f1_test,ORG_f1_test,learning_rate,batch_size
30,zefys2025,zefys2025,electra-base-german-europeana-cased-discriminator,5,0.83,0.85,0.85,0.88,0.69,2e-05,32
47,GermEval,zefys2025,bert-base-german-cased,4,0.78,0.81,0.89,0.82,0.59,2e-05,64
78,europeana-onb,zefys2025,electra-base-german-europeana-cased-discriminator,5,0.72,0.85,0.61,0.82,0.09,2e-05,32
79,hisgerman,zefys2025,electra-base-german-europeana-cased-discriminator,5,0.72,0.84,0.69,0.77,0.11,4e-05,96
107,neiss-arendt,zefys2025,bert-base-historic-multilingual-cased,6,0.68,0.83,0.79,0.79,0.07,4e-05,64
121,neiss-sturm,zefys2025,xlm-roberta-base,6,0.67,0.80,0.71,0.59,0.00,2e-05,64
171,hipe2020,zefys2025,electra-base-german-europeana-cased-discriminator,6,0.63,0.84,0.45,0.78,0.42,2e-05,64
192,conll2003,zefys2025,xlm-roberta-base,7,0.62,0.81,0.84,0.66,0.41,2e-05,32
214,europeana-lft,zefys2025,bert-base-historic-multilingual-cased,4,0.61,0.81,0.62,0.67,0.46,4e-05,96


# ZEFYS2025 Pre-Training Configs

In [9]:
atab = []
for dataset, res in load_results('experiments/zefys-pretrained/pretrained-on-zefys-configs.pkl').groupby(['test', 'model']):
    atab.append(res.sort_values('f1_test', ascending=False).iloc[[0]])
atab = pd.concat(atab).sort_values(['test','f1_early'], ascending=[True, False]).reset_index(drop=True)

zefys_pretrained_configs_table, df_zefys_pretrained_configs = make_nice_table(atab, ['zefys2025'], ['f1_test', 'f1_early',], [], additional_columns=['dataset','model','learning_rate'])
zefys_pretrained_configs_table

Number of results so far:  9


model,learning_rate,f1_test,f1_early
__Anon2025__,nan,-,-
Europ-Electra,2e-05,0.83,0.85
GermanBERT,2e-05,0.83,0.83
hmBERT,2e-05,0.81,0.83
GBERT,2e-05,0.80,0.82
XLM-Roberta,2e-05,0.81,0.81
Roberta,2e-05,0.75,0.76
Distilbert,2e-05,0.74,0.74
hmBERT-mini,2e-05,0.68,0.68
hmBERT-tiny,2e-05,0.49,0.51


# All-Historical Pre-Training Configs

In [10]:
atab = []
for dataset, res in load_results('experiments/all-historic-pretrained/pretrained-on-all-historic-configs.pkl').groupby(['test', 'model']):
    atab.append(res.sort_values('f1_test', ascending=False).iloc[[0]])
atab = pd.concat(atab).sort_values(['test','f1_early'], ascending=[True, False]).reset_index(drop=True)

all_historic_pretrained_configs_table, df_all_historic_pretrained_configs = make_nice_table(atab, ['all-historic'], ['f1_test', 'f1_early',], [], 
                                                dataset_column='train', additional_columns =['dataset', 'test', 'model', 'learning_rate', 'batch_size'])
all_historic_pretrained_configs_table

Number of results so far:  63


test,model,learning_rate,batch_size,f1_test,f1_early
__all-historic__,nan,nan,nan,-,-
Anon2025,Europ-Electra,2e-05,32,0.83,0.79
Anon2025,XLM-Roberta,2e-05,32,0.79,0.77
Anon2025,hmBERT,2e-05,32,0.80,0.76
Anon2025,GBERT,2e-05,32,0.74,0.75
Anon2025,GermanBERT,2e-05,32,0.78,0.74
Anon2025,Roberta,2e-05,32,0.75,0.71
Anon2025,hmBERT-mini,2e-05,32,0.67,0.64
Anon2025,Distilbert,2e-05,32,0.67,0.64
Anon2025,hmBERT-tiny,2e-05,32,0.54,0.53


## pretrained-on-zefys

In [11]:
atab = []
for dataset, res in load_results('experiments/results-pretrained-zefys.pkl').groupby(['train', 'test','pretrain', 'model']):
    atab.append(res.sort_values('f1_test', ascending=False).iloc[[0]])
atab = pd.concat(atab).sort_values(['test','f1_early'], ascending=[True, False]).reset_index(drop=True)
res_pretrained_zefys2=atab
atab[['test', 'pretrain', 'model', 'epoch', 'f1_test', 'f1_early', 'PER_f1_test','LOC_f1_test', 'ORG_f1_test', 'learning_rate', 'batch_size']]

Number of results so far:  486


,test,pretrain,model,epoch,f1_test,f1_early,PER_f1_test,LOC_f1_test,ORG_f1_test,learning_rate,batch_size
0,europeana-lft,zefys2025,electra-base-german-europeana-cased-discriminator,5,0.80,0.79,0.83,0.81,0.67,4e-05,32
1,europeana-lft,zefys2025,xlm-roberta-base,4,0.78,0.78,0.81,0.79,0.70,4e-05,64
2,europeana-lft,zefys2025,bert-base-historic-multilingual-cased,3,0.76,0.77,0.77,0.77,0.68,0.0001,32
3,europeana-lft,zefys2025,bert-base-german-cased,8,0.73,0.76,0.76,0.76,0.59,4e-05,32
4,europeana-lft,zefys2025,gbert-base,4,0.73,0.75,0.76,0.75,0.60,0.0001,64
5,europeana-lft,zefys2025,roberta-base,10,0.75,0.75,0.75,0.78,0.65,4e-05,96
6,europeana-lft,zefys2025,distilbert-base-uncased,5,0.67,0.69,0.63,0.73,0.59,0.0001,64
7,europeana-lft,zefys2025,bert-mini-historic-multilingual-cased,7,0.67,0.67,0.66,0.72,0.53,0.0001,96
8,europeana-lft,zefys2025,bert-tiny-historic-multilingual-cased,15,0.55,0.58,0.53,0.63,0.39,0.0001,64
9,europeana-onb,zefys2025,xlm-roberta-base,9,0.88,0.85,0.85,0.92,0.13,0.0001,32


## pretrained-on-all-historical

In [12]:
atab = []
for dataset, res in load_results('experiments/results-pretrained-all-historic.pkl').groupby(['train', 'test','pretrain', 'model']):
    atab.append(res.sort_values('f1_test', ascending=False).iloc[[0]])
atab = pd.concat(atab).sort_values(['test','f1_early'], ascending=[True, False]).reset_index(drop=True)
res_pretrained_all2=atab
atab[['test', 'pretrain', 'model', 'epoch', 'f1_test', 'f1_early', 'PER_f1_test','LOC_f1_test', 'ORG_f1_test', 'learning_rate', 'batch_size']]

Number of results so far:  547


,test,pretrain,model,epoch,f1_test,f1_early,PER_f1_test,LOC_f1_test,ORG_f1_test,learning_rate,batch_size
0,europeana-lft,all-historic,xlm-roberta-base,4,0.78,0.80,0.81,0.80,0.66,2e-05,96
1,europeana-lft,all-historic,bert-base-historic-multilingual-cased,4,0.77,0.78,0.79,0.79,0.68,2e-05,96
2,europeana-lft,all-historic,electra-base-german-europeana-cased-discriminator,4,0.79,0.78,0.82,0.82,0.66,2e-05,32
3,europeana-lft,all-historic,gbert-base,1,0.73,0.76,0.74,0.75,0.62,4e-05,64
4,europeana-lft,all-historic,roberta-base,5,0.76,0.76,0.78,0.79,0.66,4e-05,64
5,europeana-lft,all-historic,bert-base-german-cased,1,0.74,0.74,0.77,0.75,0.62,0.0001,64
6,europeana-lft,all-historic,distilbert-base-uncased,3,0.66,0.70,0.62,0.73,0.56,4e-05,32
7,europeana-lft,all-historic,bert-mini-historic-multilingual-cased,4,0.65,0.67,0.67,0.70,0.51,0.0001,96
8,europeana-lft,all-historic,bert-tiny-historic-multilingual-cased,9,0.58,0.60,0.57,0.64,0.44,0.0001,32
9,europeana-onb,all-historic,xlm-roberta-base,3,0.86,0.86,0.79,0.92,0.65,0.0001,64


In [13]:
merged_results2 = merge_historical_results(res_single, res_pretrained_zefys2, res_pretrained_all2)
merged_results2.columns

Index(['precision', 'recall', 'f1', 'accuracy', 'f1_early', 'epoch', 'model',
       'train', 'train_params', 'f1_test', 'precision_test', 'recall_test',
       'PER_f1_best', 'PER_precision_test', 'PER_recall_test', 'LOC_f1_best',
       'LOC_precision_test', 'LOC_recall_test', 'ORG_f1_best',
       'ORG_precision_test', 'ORG_recall_test', 'test', 'exp_ID', 'batch_size',
       'learning_rate', 'pretrain', 'pretrained_model'],
      dtype='object')


Index(['precision', 'recall', 'f1', 'accuracy', 'f1_early', 'epoch', 'model',
       'train', 'train_params', 'f1_test', 'precision_test', 'recall_test',
       'PER_f1_test', 'PER_precision_test', 'PER_recall_test', 'LOC_f1_test',
       'LOC_precision_test', 'LOC_recall_test', 'ORG_f1_test',
       'ORG_precision_test', 'ORG_recall_test', 'test', 'exp_ID', 'batch_size',
       'learning_rate', 'precision_PZ', 'recall_PZ', 'f1_PZ', 'accuracy_PZ',
       'f1_early_PZ', 'epoch_PZ', 'train_params_PZ', 'f1_test_PZ',
       'precision_test_PZ', 'recall_test_PZ', 'PER_f1_test_PZ',
       'PER_precision_test_PZ', 'PER_recall_test_PZ', 'LOC_f1_test_PZ',
       'LOC_precision_test_PZ', 'LOC_recall_test_PZ', 'ORG_f1_test_PZ',
       'ORG_precision_test_PZ', 'ORG_recall_test_PZ', 'pretrained_model',
       'exp_ID_PZ', 'batch_size_PZ', 'learning_rate_PZ', 'precision_PA',
       'recall_PA', 'f1_PA', 'accuracy_PA', 'f1_early_PA', 'epoch_PA',
       'train_params_PA', 'f1_test_PA', 'precision_test

#### Zefys2025 Results without Pretraining/with Pretraining on All-Historical

In [14]:
newdataset_table2, df_newdataset2 = make_nice_table(merged_results2,
                                                  ['zefys2025'], ['f1_test', 'f1_test_PA', 'PER_f1_best', 'LOC_f1_best', 'ORG_f1_best'], 
                                                  ['f1_test', 'f1_test_PA'])
newdataset_table2

model,f1_test,f1_test_PA,PER_f1_best,LOC_f1_best,ORG_f1_best
__Anon2025__,-,-,-,-,-
Europ-Electra,0.83,0.84,0.84,0.89,0.70
hmBERT,0.83,0.83,0.85,0.88,0.71
GermanBERT,0.82,0.83,0.84,0.89,0.70
GBERT,0.82,0.81,0.83,0.88,0.67
XLM-Roberta,0.80,0.81,0.82,0.87,0.67
Roberta,0.78,0.80,0.80,0.87,0.69
Distilbert,0.75,0.75,0.74,0.82,0.64
hmBERT-mini,0.72,0.72,0.70,0.80,0.57
hmBERT-tiny,0.60,0.64,0.61,0.74,0.46


In [15]:
write_to_latex_file(newdataset_table2,'tables/newdataset.tex', df_newdataset2.shape[1])

'\\begin{tabular}{lrrrrr}\n\\toprule\nmodel & $f1_{test}$ & $f1_{test}_{PA}$ & $PER_{f1}_{best}$ & $LOC_{f1}_{best}$ & $ORG_{f1}_{best}$ \\\\\n\\midrule\n\\multicolumn{6}{c}{Anon2025} \\\\\nEurop-Electra & 0.83 & \\bf 0.84 & 0.84 & 0.89 & 0.70 \\\\\nhmBERT & \\bf 0.83 & \\bf 0.83 & 0.85 & 0.88 & 0.71 \\\\\nGermanBERT & 0.82 & \\bf 0.83 & 0.84 & 0.89 & 0.70 \\\\\nGBERT & \\bf 0.82 & 0.81 & 0.83 & 0.88 & 0.67 \\\\\nXLM-Roberta & 0.80 & \\bf 0.81 & 0.82 & 0.87 & 0.67 \\\\\nRoberta & 0.78 & \\bf 0.80 & 0.80 & 0.87 & 0.69 \\\\\nDistilbert & \\bf 0.75 & \\bf 0.75 & 0.74 & 0.82 & 0.64 \\\\\nhmBERT-mini & \\bf 0.72 & \\bf 0.72 & 0.70 & 0.80 & 0.57 \\\\\nhmBERT-tiny & 0.60 & \\bf 0.64 & 0.61 & 0.74 & 0.46 \\\\\n\\bottomrule\n\\end{tabular}\n'

# Historical Datasets Results without Pretraining/with Pretraining on ZEFYS2025/with Pretraining on All-Historical

In [16]:
hist_table2, df_hist_table2 = make_nice_table(merged_results2,
                                            ['hisgerman', 'europeana-lft', 'europeana-onb', 'hipe2020', 'neiss-arendt', 'neiss-sturm'],
                                            ['f1_test', 'f1_test_PZ', 'f1_test_PA', 'PER_f1_best', 'LOC_f1_best', 'ORG_f1_best'], 
                                            ['f1_test', 'f1_test_PZ', 'f1_test_PA'])
hist_table2

model,f1_test,f1_test_PZ,f1_test_PA,PER_f1_best,LOC_f1_best,ORG_f1_best
__HisGermanNER__,-,-,-,-,-,-
hmBERT,0.80,0.82,0.84,0.88,0.84,0.21
Europ-Electra,0.80,0.84,0.81,0.86,0.86,0.15
GBERT,0.78,0.80,0.81,0.84,0.84,0.00
GermanBERT,0.77,0.79,0.82,0.81,0.87,0.16
XLM-Roberta,0.74,0.79,0.82,0.82,0.85,0.17
Roberta,0.71,0.76,0.79,0.81,0.82,0.00
Distilbert,0.61,0.67,0.74,0.72,0.78,0.10
hmBERT-mini,0.60,0.68,0.72,0.73,0.76,0.00
hmBERT-tiny,0.30,0.51,0.58,0.51,0.67,0.00


In [17]:
write_to_latex_file(hist_table2,'tables/historical_comparison.tex', df_hist_table2.shape[1])

'\\begin{tabular}{lrrrrrr}\n\\toprule\nmodel & $f1_{test}$ & $f1_{test}_{PZ}$ & $f1_{test}_{PA}$ & $PER_{f1}_{best}$ & $LOC_{f1}_{best}$ & $ORG_{f1}_{best}$ \\\\\n\\midrule\n\\multicolumn{7}{c}{HisGermanNER} \\\\\nhmBERT & 0.80 & 0.82 & \\bf 0.84 & 0.88 & 0.84 & 0.21 \\\\\nEurop-Electra & 0.80 & \\bf 0.84 & 0.81 & 0.86 & 0.86 & 0.15 \\\\\nGBERT & 0.78 & 0.80 & \\bf 0.81 & 0.84 & 0.84 & 0.00 \\\\\nGermanBERT & 0.77 & 0.79 & \\bf 0.82 & 0.81 & 0.87 & 0.16 \\\\\nXLM-Roberta & 0.74 & 0.79 & \\bf 0.82 & 0.82 & 0.85 & 0.17 \\\\\nRoberta & 0.71 & 0.76 & \\bf 0.79 & 0.81 & 0.82 & 0.00 \\\\\nDistilbert & 0.61 & 0.67 & \\bf 0.74 & 0.72 & 0.78 & 0.10 \\\\\nhmBERT-mini & 0.60 & 0.68 & \\bf 0.72 & 0.73 & 0.76 & 0.00 \\\\\nhmBERT-tiny & 0.30 & 0.51 & \\bf 0.58 & 0.51 & 0.67 & 0.00 \\\\\n\\multicolumn{7}{c}{Europeana-LFT} \\\\\nEurop-Electra & 0.79 & \\bf 0.80 & 0.79 & 0.83 & 0.81 & 0.67 \\\\\nXLM-Roberta & \\bf 0.78 & \\bf 0.78 & \\bf 0.78 & 0.80 & 0.80 & 0.68 \\\\\nhmBERT & 0.76 & 0.76 & \\bf 0.77 